# MO-PPO改进算法科研论文分析

## 实验设置

- **改进算法**：MO-PPO2, MO-PPO2E, MO-IPPO（基于MO-PPO的改进）
- **基线算法**：MO-PPO
- **对比算法**：MO-WOA, MO-DBO, MO-HHO, MO-GWO, MO-SFOA, MO-Sequoia, NSGA-II, MOEA/D, SPEA2
- **实验轮次**：run_21 到 run_30（共10次独立运行）
- **任务规模**：
  - run_21: 100个任务
  - run_22: 200个任务
  - run_23: 300个任务
  - run_24: 400个任务
  - run_25: 500个任务
  - run_26: 600个任务
  - run_27: 700个任务
  - run_28: 800个任务
  - run_29: 900个任务
  - run_30: 1000个任务
- **评估指标**：
  - 单目标：Makespan, TotalTime, LoadBalance, Cost
  - 多目标：Hypervolume (HV), Generational Distance (GD), Inverted Generational Distance (IGD), Spread
- **分析维度**：
  - 整体性能对比（所有任务规模）
  - 可扩展性分析（不同任务规模下的性能变化）
  - 任务规模对算法性能的影响

In [ ]:
# ==================== 导入库 ====================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import friedmanchisquare, wilcoxon
from itertools import combinations
import os
import warnings
warnings.filterwarnings('ignore')

# ==================== 设置中文字体（修复中文乱码） ====================
import matplotlib
from matplotlib import font_manager

# 尝试设置中文字体
def setup_chinese_font():
    """设置中文字体，解决中文乱码问题"""
    # Windows系统常见中文字体
    windows_fonts = ['SimHei', 'Microsoft YaHei', 'KaiTi', 'FangSong', 'SimSun']
    # macOS系统常见中文字体
    mac_fonts = ['Arial Unicode MS', 'STHeiti', 'STSong', 'Heiti TC', 'PingFang SC']
    # Linux系统常见中文字体
    linux_fonts = ['WenQuanYi Micro Hei', 'WenQuanYi Zen Hei', 'Noto Sans CJK SC']
    
    # 获取系统所有可用字体
    available_fonts = [f.name for f in font_manager.fontManager.ttflist]
    
    # 按优先级查找可用字体
    font_candidates = windows_fonts + mac_fonts + linux_fonts
    
    found_font = None
    for font in font_candidates:
        if font in available_fonts:
            found_font = font
            break
    
    if found_font:
        plt.rcParams['font.sans-serif'] = [found_font] + plt.rcParams['font.sans-serif']
        print(f"✅ 使用中文字体: {found_font}")
    else:
        # 如果没有找到中文字体，尝试使用matplotlib的默认设置
        # 并给出警告
        print("⚠️ 警告：未找到中文字体，可能显示为方块")
        print("   可用字体列表（前10个）:", available_fonts[:10])
        print("   建议安装中文字体或使用以下代码手动设置：")
        print("   plt.rcParams['font.sans-serif'] = ['你的中文字体名称']")
        
        # 尝试使用DejaVu Sans作为备用（虽然不支持中文，但至少不会报错）
        plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
    
    # 解决负号显示问题
    plt.rcParams['axes.unicode_minus'] = False
    
    return found_font

# 设置中文字体
chinese_font = setup_chinese_font()

# 如果找不到中文字体，设置使用英文标签的标志
USE_ENGLISH_LABELS = chinese_font is None

if USE_ENGLISH_LABELS:
    print("ℹ️ 将使用英文标签替代中文标签")

# 设置绘图风格
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

print("✅ 库导入完成")

In [ ]:
# ==================== 配置参数 ====================

# 实验目录范围
RUN_START = 21
RUN_END = 30
RUNS = list(range(RUN_START, RUN_END + 1))

# 任务规模映射：run编号 -> 任务数
TASK_SIZE_MAP = {run_num: (run_num - RUN_START + 1) * 100 for run_num in RUNS}
# {21: 100, 22: 200, 23: 300, ..., 30: 1000}

# ==================== 标签映射（根据字体可用性选择中文或英文） ====================
# 如果找不到中文字体，使用英文标签
if USE_ENGLISH_LABELS:
    LABELS = {
        'task_number': 'Task Number',
        'improvement_rate': 'Improvement Rate (%)',
        'value': 'Value',
        'metric': 'Metric',
        'algorithm': 'Algorithm',
        'performance_heatmap': 'Algorithm Performance Heatmap (Normalized, Lower is Better)',
    }
    def scalability_title(metric): return f'{metric} vs Task Size'
    def improvement_title(metric): return f'{metric} Improvement Rate vs Task Size'
else:
    LABELS = {
        'task_number': '任务数量',
        'improvement_rate': '改进率 (%)',
        'value': 'Value',
        'metric': '指标',
        'algorithm': '算法',
        'performance_heatmap': '算法性能热力图（归一化值，越小越好）',
    }
    def scalability_title(metric): return f'{metric} 随任务规模的变化'
    def improvement_title(metric): return f'{metric} 改进率随任务规模的变化'

# 算法分类
PROPOSED_ALGORITHMS = {
    'MOPPO': {'name': 'MO-PPO', 'type': 'baseline', 'color': '#1f77b4', 'marker': 'o'},
    'MOPPO2': {'name': 'MO-PPO2', 'type': 'proposed', 'color': '#ff7f0e', 'marker': 's'},
    'MOPPO2E': {'name': 'MO-PPO2E', 'type': 'proposed', 'color': '#2ca02c', 'marker': '^'},
    'MOIPPO': {'name': 'MO-IPPO', 'type': 'proposed', 'color': '#d62728', 'marker': 'D'}
}

COMPARISON_ALGORITHMS = {
    'MOWOA': {'name': 'MO-WOA', 'color': '#9467bd', 'marker': 'p'},
    'MODBO': {'name': 'MO-DBO', 'color': '#8c564b', 'marker': '*'},
    'MOHHO': {'name': 'MO-HHO', 'color': '#e377c2', 'marker': 'h'},
    'MOGWO': {'name': 'MO-GWO', 'color': '#7f7f7f', 'marker': 'X'},
    'MOSFOA': {'name': 'MO-SFOA', 'color': '#bcbd22', 'marker': '+'},
    'mosequoia': {'name': 'MO-Sequoia', 'color': '#17becf', 'marker': '1'},
    'NSGAII': {'name': 'NSGA-II', 'color': '#ff9896', 'marker': '2'},
    'MOEAD': {'name': 'MOEA/D', 'color': '#c5b0d5', 'marker': '3'},
    'SPEA2': {'name': 'SPEA2', 'color': '#c49c94', 'marker': '4'}
}

ALL_ALGORITHMS = {**PROPOSED_ALGORITHMS, **COMPARISON_ALGORITHMS}

# 单目标指标
SINGLE_OBJECTIVE_METRICS = ['Makespan', 'TotalTime', 'LoadBalance', 'Cost']

# 多目标指标
MULTI_OBJECTIVE_METRICS = ['HV', 'GD', 'IGD', 'Spread']

# 试验次数
NUM_TRIALS = 10

print(f"✅ 配置完成：分析 run_{RUN_START} 到 run_{RUN_END}，共 {len(RUNS)} 次运行")

In [ ]:
# ==================== 辅助函数：多目标指标计算 ====================

def compute_hypervolume(pareto_front, reference_point=None):
    """
    计算超体积（Hypervolume）
    pareto_front: numpy array, shape (n_solutions, n_objectives)
    reference_point: 参考点，如果为None则使用Pareto前沿的最大值
    """
    if len(pareto_front) == 0:
        return 0.0
    
    pareto_front = np.array(pareto_front)
    n_obj = pareto_front.shape[1]
    
    # 如果没有提供参考点，使用Pareto前沿的最大值
    if reference_point is None:
        reference_point = np.max(pareto_front, axis=0) * 1.1
    
    # 确保所有解都在参考点下方
    pareto_front = np.minimum(pareto_front, reference_point)
    
    # 简化版HV计算（使用蒙特卡洛方法或精确计算）
    # 这里使用简化的矩形体积计算
    if n_obj == 2:
        # 2D情况：计算面积
        sorted_front = pareto_front[np.argsort(pareto_front[:, 0])]
        hv = 0.0
        prev_x = reference_point[0]
        for point in sorted_front:
            hv += (prev_x - point[0]) * (reference_point[1] - point[1])
            prev_x = point[0]
        return hv
    else:
        # 多维情况：使用近似方法
        # 计算所有解到参考点形成的超体积
        hv = 0.0
        for point in pareto_front:
            volume = np.prod(reference_point - point)
            hv += volume / len(pareto_front)  # 简化处理
        return hv

def compute_gd(pareto_front, true_pareto_front):
    """
    计算世代距离（Generational Distance）
    pareto_front: 算法得到的Pareto前沿
    true_pareto_front: 真实Pareto前沿（使用所有算法的最优解合并）
    """
    if len(pareto_front) == 0:
        return np.inf
    
    pareto_front = np.array(pareto_front)
    true_pareto_front = np.array(true_pareto_front)
    
    distances = []
    for point in pareto_front:
        min_dist = np.min(np.linalg.norm(true_pareto_front - point, axis=1))
        distances.append(min_dist)
    
    return np.mean(distances)

def compute_igd(pareto_front, true_pareto_front):
    """
    计算反向世代距离（Inverted Generational Distance）
    """
    if len(true_pareto_front) == 0:
        return np.inf
    
    pareto_front = np.array(pareto_front)
    true_pareto_front = np.array(true_pareto_front)
    
    distances = []
    for point in true_pareto_front:
        min_dist = np.min(np.linalg.norm(pareto_front - point, axis=1))
        distances.append(min_dist)
    
    return np.mean(distances)

def compute_spread(pareto_front):
    """
    计算Spread指标（多样性指标）
    """
    if len(pareto_front) < 2:
        return np.inf
    
    pareto_front = np.array(pareto_front)
    n_obj = pareto_front.shape[1]
    
    # 计算极端点
    extreme_points = []
    for i in range(n_obj):
        idx = np.argmin(pareto_front[:, i])
        extreme_points.append(pareto_front[idx])
    
    # 计算相邻解之间的距离
    distances = []
    sorted_indices = np.argsort(pareto_front[:, 0])
    sorted_front = pareto_front[sorted_indices]
    
    for i in range(len(sorted_front) - 1):
        dist = np.linalg.norm(sorted_front[i+1] - sorted_front[i])
        distances.append(dist)
    
    if len(distances) == 0:
        return np.inf
    
    mean_dist = np.mean(distances)
    
    # 计算到极端点的距离
    df = np.linalg.norm(sorted_front[0] - extreme_points[0])
    dl = np.linalg.norm(sorted_front[-1] - extreme_points[-1])
    
    # Spread公式
    spread = (df + dl + np.sum(np.abs(distances - mean_dist))) / (df + dl + len(distances) * mean_dist)
    
    return spread

def normalize_objectives(pareto_front):
    """归一化目标值"""
    pareto_front = np.array(pareto_front)
    if len(pareto_front) == 0:
        return pareto_front
    
    # 使用最小-最大归一化
    min_vals = np.min(pareto_front, axis=0)
    max_vals = np.max(pareto_front, axis=0)
    
    # 避免除零
    ranges = max_vals - min_vals
    ranges[ranges == 0] = 1.0
    
    normalized = (pareto_front - min_vals) / ranges
    return normalized

print("✅ 多目标指标计算函数定义完成")

In [ ]:
# ==================== 数据加载 ====================

def load_single_objective_data(run_num):
    """加载单目标数据"""
    data = {}
    base_dir = f"../results/run_{run_num}"
    
    for alg_name in ALL_ALGORITHMS.keys():
        file_path = os.path.join(base_dir, f"{alg_name}.csv")
        if os.path.exists(file_path):
            df = pd.read_csv(file_path)
            data[alg_name] = df
        else:
            print(f"⚠️ 警告：{file_path} 不存在")
    
    return data

def load_pareto_front(run_num, alg_name, trial, generation='final'):
    """加载Pareto前沿数据"""
    base_dir = f"../results/run_{run_num}/pareto_fronts"
    file_path = os.path.join(base_dir, f"{alg_name}_trial_{trial}_{generation}.csv")
    
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        return df.values
    else:
        return None

# 加载所有数据
print("📊 开始加载数据...")
all_single_obj_data = {}
all_pareto_data = {}

for run_num in RUNS:
    print(f"  加载 run_{run_num}...")
    all_single_obj_data[run_num] = load_single_objective_data(run_num)
    
    # 加载Pareto前沿数据
    pareto_data = {}
    for alg_name in ALL_ALGORITHMS.keys():
        alg_pareto = {}
        for trial in range(1, NUM_TRIALS + 1):
            final_pf = load_pareto_front(run_num, alg_name, trial, 'final')
            first_pf = load_pareto_front(run_num, alg_name, trial, 'first')
            if final_pf is not None:
                alg_pareto[trial] = {'final': final_pf, 'first': first_pf}
        if alg_pareto:
            pareto_data[alg_name] = alg_pareto
    all_pareto_data[run_num] = pareto_data

print(f"✅ 数据加载完成：{len(RUNS)} 次运行")

In [ ]:
# ==================== 合并所有运行的数据 ====================

# 合并单目标数据（保留任务规模信息）
merged_single_obj = {}
for alg_name in ALL_ALGORITHMS.keys():
    all_trials = []
    for run_num in RUNS:
        if alg_name in all_single_obj_data[run_num]:
            df = all_single_obj_data[run_num][alg_name].copy()
            df['Run'] = run_num
            df['TaskSize'] = TASK_SIZE_MAP[run_num]  # 添加任务规模列
            all_trials.append(df)
    
    if all_trials:
        merged_single_obj[alg_name] = pd.concat(all_trials, ignore_index=True)

print(f"✅ 单目标数据合并完成，共 {len(merged_single_obj)} 个算法")
print(f"   任务规模范围：{min(TASK_SIZE_MAP.values())} - {max(TASK_SIZE_MAP.values())} 个任务")

# 合并Pareto前沿数据
merged_pareto = {}
for alg_name in ALL_ALGORITHMS.keys():
    alg_data = {}
    trial_counter = 1
    for run_num in RUNS:
        if alg_name in all_pareto_data[run_num]:
            for trial in range(1, NUM_TRIALS + 1):
                if trial in all_pareto_data[run_num][alg_name]:
                    alg_data[trial_counter] = all_pareto_data[run_num][alg_name][trial]
                    trial_counter += 1
    if alg_data:
        merged_pareto[alg_name] = alg_data

print(f"✅ Pareto前沿数据合并完成，共 {len(merged_pareto)} 个算法")

In [ ]:
# ==================== 计算多目标指标 ====================

# 首先构建真实Pareto前沿（所有算法所有运行的最优解合并）
print("🔍 构建真实Pareto前沿...")
all_pareto_solutions = []
for alg_name in merged_pareto.keys():
    for trial in merged_pareto[alg_name].keys():
        final_pf = merged_pareto[alg_name][trial]['final']
        if final_pf is not None and len(final_pf) > 0:
            all_pareto_solutions.extend(final_pf.tolist())

all_pareto_solutions = np.array(all_pareto_solutions)

# 提取非支配解作为真实Pareto前沿
def get_non_dominated(solutions):
    """获取非支配解"""
    if len(solutions) == 0:
        return solutions
    
    solutions = np.array(solutions)
    n = len(solutions)
    is_non_dominated = np.ones(n, dtype=bool)
    
    for i in range(n):
        if not is_non_dominated[i]:
            continue
        for j in range(n):
            if i == j:
                continue
            if not is_non_dominated[j]:
                continue
            # 检查j是否支配i
            if np.all(solutions[j] <= solutions[i]) and np.any(solutions[j] < solutions[i]):
                is_non_dominated[i] = False
                break
    
    return solutions[is_non_dominated]

true_pareto_front = get_non_dominated(all_pareto_solutions)
print(f"✅ 真实Pareto前沿构建完成，包含 {len(true_pareto_front)} 个非支配解")

# 计算参考点（用于HV计算）
reference_point = np.max(true_pareto_front, axis=0) * 1.1
print(f"📌 参考点: {reference_point}")

# 计算每个算法的多目标指标
multi_obj_results = {}
for alg_name in merged_pareto.keys():
    alg_metrics = {'HV': [], 'GD': [], 'IGD': [], 'Spread': []}
    
    for trial in merged_pareto[alg_name].keys():
        final_pf = merged_pareto[alg_name][trial]['final']
        if final_pf is not None and len(final_pf) > 0:
            # 归一化目标值（用于指标计算）
            normalized_pf = normalize_objectives(final_pf)
            normalized_true_pf = normalize_objectives(true_pareto_front)
            normalized_ref = normalize_objectives(reference_point.reshape(1, -1))[0]
            
            # 计算指标
            hv = compute_hypervolume(normalized_pf, normalized_ref)
            gd = compute_gd(normalized_pf, normalized_true_pf)
            igd = compute_igd(normalized_pf, normalized_true_pf)
            spread = compute_spread(normalized_pf)
            
            alg_metrics['HV'].append(hv)
            alg_metrics['GD'].append(gd)
            alg_metrics['IGD'].append(igd)
            alg_metrics['Spread'].append(spread)
    
    multi_obj_results[alg_name] = alg_metrics

print(f"✅ 多目标指标计算完成，共 {len(multi_obj_results)} 个算法")

## 1. 描述性统计分析

In [ ]:
# ==================== 单目标指标描述性统计 ====================

def compute_statistics(data_dict, metric):
    """计算统计量"""
    stats_dict = {}
    for alg_name, df in data_dict.items():
        if metric in df.columns:
            values = df[metric].values
            stats_dict[alg_name] = {
                'Mean': np.mean(values),
                'Std': np.std(values),
                'Min': np.min(values),
                'Max': np.max(values),
                'Median': np.median(values),
                'Q1': np.percentile(values, 25),
                'Q3': np.percentile(values, 75)
            }
    return stats_dict

# 计算所有单目标指标的统计量
single_obj_stats = {}
for metric in SINGLE_OBJECTIVE_METRICS:
    single_obj_stats[metric] = compute_statistics(merged_single_obj, metric)

# 显示统计结果
print("=" * 80)
print("单目标指标描述性统计")
print("=" * 80)

for metric in SINGLE_OBJECTIVE_METRICS:
    print(f"\n【{metric}】")
    print("-" * 80)
    df_stats = pd.DataFrame(single_obj_stats[metric]).T
    df_stats = df_stats.round(2)
    print(df_stats.to_string())

In [ ]:
# ==================== 多目标指标描述性统计 ====================

multi_obj_stats = {}
for metric in MULTI_OBJECTIVE_METRICS:
    stats_dict = {}
    for alg_name in multi_obj_results.keys():
        values = multi_obj_results[alg_name][metric]
        if len(values) > 0:
            stats_dict[alg_name] = {
                'Mean': np.mean(values),
                'Std': np.std(values),
                'Min': np.min(values),
                'Max': np.max(values),
                'Median': np.median(values),
                'Q1': np.percentile(values, 25),
                'Q3': np.percentile(values, 75)
            }
    multi_obj_stats[metric] = stats_dict

# 显示统计结果
print("=" * 80)
print("多目标指标描述性统计")
print("=" * 80)

for metric in MULTI_OBJECTIVE_METRICS:
    print(f"\n【{metric}】")
    print("-" * 80)
    df_stats = pd.DataFrame(multi_obj_stats[metric]).T
    df_stats = df_stats.round(4)
    print(df_stats.to_string())

## 2. 统计显著性检验

In [ ]:
# ==================== Wilcoxon符号秩检验（改进算法 vs 基线算法） ====================

def wilcoxon_test(data1, data2, alpha=0.05):
    """执行Wilcoxon符号秩检验"""
    try:
        statistic, p_value = wilcoxon(data1, data2, alternative='two-sided')
        return {
            'statistic': statistic,
            'p_value': p_value,
            'significant': p_value < alpha,
            'better': 'data1' if np.mean(data1) < np.mean(data2) else 'data2'
        }
    except:
        return None

# 对每个改进算法与基线算法进行Wilcoxon检验
print("=" * 80)
print("Wilcoxon符号秩检验：改进算法 vs MO-PPO（基线）")
print("=" * 80)

baseline_alg = 'MOPPO'
improved_algs = ['MOPPO2', 'MOPPO2E', 'MOIPPO']

wilcoxon_results = {}

for metric in SINGLE_OBJECTIVE_METRICS:
    print(f"\n【{metric}】")
    print("-" * 80)
    
    if baseline_alg in merged_single_obj:
        baseline_data = merged_single_obj[baseline_alg][metric].values
        
        for improved_alg in improved_algs:
            if improved_alg in merged_single_obj:
                improved_data = merged_single_obj[improved_alg][metric].values
                
                # 确保数据长度一致
                min_len = min(len(baseline_data), len(improved_data))
                baseline_subset = baseline_data[:min_len]
                improved_subset = improved_data[:min_len]
                
                result = wilcoxon_test(improved_subset, baseline_subset)
                
                if result:
                    improvement = ((np.mean(baseline_subset) - np.mean(improved_subset)) / np.mean(baseline_subset)) * 100
                    sign = "***" if result['significant'] else ""
                    print(f"{ALL_ALGORITHMS[improved_alg]['name']} vs {ALL_ALGORITHMS[baseline_alg]['name']}: "
                          f"p={result['p_value']:.4f} {sign}, "
                          f"改进率={improvement:.2f}%")
                    
                    if metric not in wilcoxon_results:
                        wilcoxon_results[metric] = {}
                    wilcoxon_results[metric][improved_alg] = result

In [ ]:
# ==================== Friedman检验（所有算法排名） ====================

def friedman_test_all_algorithms(metric):
    """对所有算法进行Friedman检验"""
    # 准备数据矩阵：行为试验，列为算法
    data_matrix = []
    alg_names = []
    
    for alg_name in ALL_ALGORITHMS.keys():
        if alg_name in merged_single_obj and metric in merged_single_obj[alg_name].columns:
            values = merged_single_obj[alg_name][metric].values
            if len(values) > 0:
                data_matrix.append(values)
                alg_names.append(alg_name)
    
    if len(data_matrix) < 2:
        return None
    
    # 确保所有算法有相同数量的试验
    min_len = min(len(arr) for arr in data_matrix)
    data_matrix = [arr[:min_len] for arr in data_matrix]
    data_matrix = np.array(data_matrix).T  # 转置：行为试验，列为算法
    
    # 计算排名
    ranks = np.argsort(np.argsort(data_matrix, axis=1), axis=1) + 1
    mean_ranks = np.mean(ranks, axis=0)
    
    # Friedman检验
    try:
        statistic, p_value = friedmanchisquare(*data_matrix.T)
        return {
            'statistic': statistic,
            'p_value': p_value,
            'mean_ranks': mean_ranks,
            'alg_names': alg_names,
            'significant': p_value < 0.05
        }
    except:
        return None

# 对所有单目标指标进行Friedman检验
print("=" * 80)
print("Friedman检验：所有算法排名")
print("=" * 80)

friedman_results = {}
for metric in SINGLE_OBJECTIVE_METRICS:
    print(f"\n【{metric}】")
    print("-" * 80)
    
    result = friedman_test_all_algorithms(metric)
    if result:
        friedman_results[metric] = result
        
        # 创建排名表
        rank_df = pd.DataFrame({
            'Algorithm': [ALL_ALGORITHMS[alg]['name'] for alg in result['alg_names']],
            'Mean Rank': result['mean_ranks'],
            'Type': [PROPOSED_ALGORITHMS.get(alg, COMPARISON_ALGORITHMS.get(alg, {}))['type'] 
                    if alg in PROPOSED_ALGORITHMS else 'comparison' for alg in result['alg_names']]
        })
        rank_df = rank_df.sort_values('Mean Rank')
        rank_df['Rank'] = range(1, len(rank_df) + 1)
        
        print(f"Friedman统计量: {result['statistic']:.4f}")
        print(f"p值: {result['p_value']:.4f}")
        print(f"显著性: {'是' if result['significant'] else '否'}")
        print("\n算法排名:")
        print(rank_df.to_string(index=False))

## 3. 可视化分析

In [ ]:
# ==================== 单目标指标箱线图 ====================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for idx, metric in enumerate(SINGLE_OBJECTIVE_METRICS):
    ax = axes[idx]
    
    # 准备数据
    plot_data = []
    plot_labels = []
    plot_colors = []
    
    # 先添加改进算法
    for alg_name in ['MOPPO', 'MOPPO2', 'MOPPO2E', 'MOIPPO']:
        if alg_name in merged_single_obj and metric in merged_single_obj[alg_name].columns:
            plot_data.append(merged_single_obj[alg_name][metric].values)
            plot_labels.append(ALL_ALGORITHMS[alg_name]['name'])
            plot_colors.append(ALL_ALGORITHMS[alg_name]['color'])
    
    # 再添加对比算法
    for alg_name in COMPARISON_ALGORITHMS.keys():
        if alg_name in merged_single_obj and metric in merged_single_obj[alg_name].columns:
            plot_data.append(merged_single_obj[alg_name][metric].values)
            plot_labels.append(ALL_ALGORITHMS[alg_name]['name'])
            plot_colors.append(ALL_ALGORITHMS[alg_name]['color'])
    
    # 绘制箱线图
    bp = ax.boxplot(plot_data, labels=plot_labels, patch_artist=True, 
                    showmeans=True, meanline=True)
    
    # 设置颜色
    for patch, color in zip(bp['boxes'], plot_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    ax.set_title(f'{metric}', fontsize=14, fontweight='bold')
    ax.set_ylabel('Value', fontsize=12)
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../draw/single_objective_boxplots.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ 单目标指标箱线图已保存")

In [ ]:
# ==================== 多目标指标箱线图 ====================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for idx, metric in enumerate(MULTI_OBJECTIVE_METRICS):
    ax = axes[idx]
    
    # 准备数据
    plot_data = []
    plot_labels = []
    plot_colors = []
    
    # 先添加改进算法
    for alg_name in ['MOPPO', 'MOPPO2', 'MOPPO2E', 'MOIPPO']:
        if alg_name in multi_obj_results:
            values = multi_obj_results[alg_name][metric]
            if len(values) > 0:
                plot_data.append(values)
                plot_labels.append(ALL_ALGORITHMS[alg_name]['name'])
                plot_colors.append(ALL_ALGORITHMS[alg_name]['color'])
    
    # 再添加对比算法
    for alg_name in COMPARISON_ALGORITHMS.keys():
        if alg_name in multi_obj_results:
            values = multi_obj_results[alg_name][metric]
            if len(values) > 0:
                plot_data.append(values)
                plot_labels.append(ALL_ALGORITHMS[alg_name]['name'])
                plot_colors.append(ALL_ALGORITHMS[alg_name]['color'])
    
    # 绘制箱线图
    if plot_data:
        bp = ax.boxplot(plot_data, labels=plot_labels, patch_artist=True, 
                        showmeans=True, meanline=True)
        
        # 设置颜色
        for patch, color in zip(bp['boxes'], plot_colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        
        ax.set_title(f'{metric}', fontsize=14, fontweight='bold')
        ax.set_ylabel('Value', fontsize=12)
        ax.tick_params(axis='x', rotation=45)
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../draw/multi_objective_boxplots.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ 多目标指标箱线图已保存")

In [ ]:
# ==================== Pareto前沿2D可视化 ====================

def plot_pareto_front_2d(ax, pareto_front, alg_name, color, marker, label):
    """绘制2D Pareto前沿"""
    if pareto_front is not None and len(pareto_front) > 0:
        # 选择前两个目标进行可视化
        ax.scatter(pareto_front[:, 0], pareto_front[:, 1], 
                  c=color, marker=marker, s=50, alpha=0.6, label=label, edgecolors='black', linewidths=0.5)

# 选择几个代表性的目标对进行可视化
objective_pairs = [
    (0, 1, 'Makespan', 'CostEfficiency'),
    (0, 2, 'Makespan', 'LoadBalanceIndex'),
    (1, 3, 'CostEfficiency', 'ResourceWaste')
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for plot_idx, (obj1_idx, obj2_idx, obj1_name, obj2_name) in enumerate(objective_pairs):
    ax = axes[plot_idx]
    
    # 绘制所有算法的Pareto前沿
    for alg_name in ['MOPPO', 'MOPPO2', 'MOPPO2E', 'MOIPPO']:
        if alg_name in merged_pareto:
            # 合并所有试验的Pareto前沿
            all_solutions = []
            for trial in merged_pareto[alg_name].keys():
                final_pf = merged_pareto[alg_name][trial]['final']
                if final_pf is not None and len(final_pf) > 0:
                    all_solutions.extend(final_pf.tolist())
            
            if all_solutions:
                all_solutions = np.array(all_solutions)
                # 提取非支配解
                non_dominated = get_non_dominated(all_solutions)
                if len(non_dominated) > 0:
                    plot_pareto_front_2d(ax, non_dominated[:, [obj1_idx, obj2_idx]], 
                                        alg_name, 
                                        ALL_ALGORITHMS[alg_name]['color'],
                                        ALL_ALGORITHMS[alg_name]['marker'],
                                        ALL_ALGORITHMS[alg_name]['name'])
    
    ax.set_xlabel(obj1_name, fontsize=12)
    ax.set_ylabel(obj2_name, fontsize=12)
    ax.set_title(f'{obj1_name} vs {obj2_name}', fontsize=14, fontweight='bold')
    ax.legend(loc='best', fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../draw/pareto_front_2d_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Pareto前沿2D可视化已保存")

In [ ]:
# ==================== 算法性能热力图 ====================

# 准备热力图数据
heatmap_data = []

for alg_name in ['MOPPO', 'MOPPO2', 'MOPPO2E', 'MOIPPO'] + list(COMPARISON_ALGORITHMS.keys()):
    if alg_name in merged_single_obj:
        row_data = []
        alg_info = []
        
        for metric in SINGLE_OBJECTIVE_METRICS:
            if metric in merged_single_obj[alg_name].columns:
                mean_val = np.mean(merged_single_obj[alg_name][metric].values)
                row_data.append(mean_val)
            else:
                row_data.append(np.nan)
        
        # 归一化（每个指标单独归一化，越小越好）
        row_data_normalized = []
        for idx, metric in enumerate(SINGLE_OBJECTIVE_METRICS):
            if not np.isnan(row_data[idx]):
                # 获取该指标的所有值
                all_values = []
                for other_alg in merged_single_obj.keys():
                    if metric in merged_single_obj[other_alg].columns:
                        all_values.extend(merged_single_obj[other_alg][metric].values)
                
                if all_values:
                    min_val = np.min(all_values)
                    max_val = np.max(all_values)
                    if max_val > min_val:
                        normalized = (row_data[idx] - min_val) / (max_val - min_val)
                    else:
                        normalized = 0.5
                    row_data_normalized.append(normalized)
                else:
                    row_data_normalized.append(0.5)
            else:
                row_data_normalized.append(0.5)
        
        heatmap_data.append(row_data_normalized)

# 创建热力图
if heatmap_data:
    heatmap_df = pd.DataFrame(heatmap_data, 
                              index=[ALL_ALGORITHMS[alg]['name'] for alg in 
                                    ['MOPPO', 'MOPPO2', 'MOPPO2E', 'MOIPPO'] + list(COMPARISON_ALGORITHMS.keys()) 
                                    if alg in merged_single_obj],
                              columns=SINGLE_OBJECTIVE_METRICS)
    
    plt.figure(figsize=(10, 12))
    sns.heatmap(heatmap_df, annot=True, fmt='.3f', cmap='RdYlGn_r', 
                cbar_kws={'label': 'Normalized Value (Lower is Better)'},
                linewidths=0.5, linecolor='gray')
    plt.title(LABELS['performance_heatmap'], fontsize=14, fontweight='bold', pad=20)
    plt.xlabel(LABELS['metric'], fontsize=12)
    plt.ylabel(LABELS['algorithm'], fontsize=12)
    plt.tight_layout()
    plt.savefig('../draw/performance_heatmap.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✅ 性能热力图已保存")

## 4. 生成论文表格

In [ ]:
# ==================== 生成单目标性能表格（CSV格式） ====================

def generate_csv_table_single_objective():
    """生成单目标性能CSV表格"""
    
    # 准备数据
    table_data = []
    
    for alg_name in ['MOPPO', 'MOPPO2', 'MOPPO2E', 'MOIPPO'] + list(COMPARISON_ALGORITHMS.keys()):
        if alg_name in merged_single_obj:
            row = {'Algorithm': ALL_ALGORITHMS[alg_name]['name']}
            
            for metric in SINGLE_OBJECTIVE_METRICS:
                if metric in merged_single_obj[alg_name].columns:
                    values = merged_single_obj[alg_name][metric].values
                    mean_val = np.mean(values)
                    std_val = np.std(values)
                    row[f'{metric}_Mean'] = mean_val
                    row[f'{metric}_Std'] = std_val
                    row[metric] = f"{mean_val:.2f} ± {std_val:.2f}"
                else:
                    row[f'{metric}_Mean'] = np.nan
                    row[f'{metric}_Std'] = np.nan
                    row[metric] = "N/A"
            
            table_data.append(row)
    
    df_table = pd.DataFrame(table_data)
    
    # 保存为CSV文件
    csv_path = '../draw/table_single_objective.csv'
    df_table.to_csv(csv_path, index=False, encoding='utf-8-sig')
    
    print("=" * 80)
    print("单目标性能表格（CSV格式）")
    print("=" * 80)
    print(f"\n✅ CSV表格已保存到 {csv_path}")
    
    return df_table

single_obj_table = generate_csv_table_single_objective()
print("\n表格预览：")
print(single_obj_table.to_string(index=False))

In [ ]:
# ==================== 生成多目标性能表格（CSV格式） ====================

def generate_csv_table_multi_objective():
    """生成多目标性能CSV表格"""
    
    # 准备数据
    table_data = []
    
    for alg_name in ['MOPPO', 'MOPPO2', 'MOPPO2E', 'MOIPPO'] + list(COMPARISON_ALGORITHMS.keys()):
        if alg_name in multi_obj_results:
            row = {'Algorithm': ALL_ALGORITHMS[alg_name]['name']}
            
            for metric in MULTI_OBJECTIVE_METRICS:
                values = multi_obj_results[alg_name][metric]
                if len(values) > 0:
                    mean_val = np.mean(values)
                    std_val = np.std(values)
                    row[f'{metric}_Mean'] = mean_val
                    row[f'{metric}_Std'] = std_val
                    row[metric] = f"{mean_val:.4f} ± {std_val:.4f}"
                else:
                    row[f'{metric}_Mean'] = np.nan
                    row[f'{metric}_Std'] = np.nan
                    row[metric] = "N/A"
            
            table_data.append(row)
    
    df_table = pd.DataFrame(table_data)
    
    # 保存为CSV文件
    csv_path = '../draw/table_multi_objective.csv'
    df_table.to_csv(csv_path, index=False, encoding='utf-8-sig')
    
    print("=" * 80)
    print("多目标性能表格（CSV格式）")
    print("=" * 80)
    print(f"\n✅ CSV表格已保存到 {csv_path}")
    
    return df_table

multi_obj_table = generate_csv_table_multi_objective()
print("\n表格预览：")
print(multi_obj_table.to_string(index=False))

In [ ]:
# ==================== 生成统计显著性检验表格（CSV格式） ====================

def generate_statistical_test_table():
    """生成统计显著性检验CSV表格"""
    
    table_data = []
    
    baseline_alg = 'MOPPO'
    improved_algs = ['MOPPO2', 'MOPPO2E', 'MOIPPO']
    
    for metric in SINGLE_OBJECTIVE_METRICS:
        if baseline_alg in merged_single_obj and metric in merged_single_obj[baseline_alg].columns:
            baseline_mean = np.mean(merged_single_obj[baseline_alg][metric].values)
            
            for improved_alg in improved_algs:
                if improved_alg in merged_single_obj and metric in merged_single_obj[improved_alg].columns:
                    improved_mean = np.mean(merged_single_obj[improved_alg][metric].values)
                    improved_std = np.std(merged_single_obj[improved_alg][metric].values)
                    
                    # 计算改进率
                    improvement = ((baseline_mean - improved_mean) / baseline_mean) * 100
                    
                    # 获取Wilcoxon检验结果
                    if metric in wilcoxon_results and improved_alg in wilcoxon_results[metric]:
                        p_value = wilcoxon_results[metric][improved_alg]['p_value']
                        significant = wilcoxon_results[metric][improved_alg]['significant']
                        sign = "***" if significant else ""
                    else:
                        p_value = np.nan
                        significant = False
                        sign = ""
                    
                    table_data.append({
                        'Metric': metric,
                        'Algorithm': ALL_ALGORITHMS[improved_alg]['name'],
                        'Mean': improved_mean,
                        'Std': improved_std,
                        'Improvement_Percent': improvement,
                        'p_value': p_value,
                        'Significant': sign
                    })
    
    df_table = pd.DataFrame(table_data)
    
    # 保存为CSV文件
    csv_path = '../draw/table_statistical_test.csv'
    df_table.to_csv(csv_path, index=False, encoding='utf-8-sig')
    
    print("=" * 80)
    print("统计显著性检验表格（CSV格式）")
    print("=" * 80)
    print(f"\n✅ CSV表格已保存到 {csv_path}")
    
    return df_table

stat_test_table = generate_statistical_test_table()
print("\n表格预览：")
print(stat_test_table.to_string(index=False))

In [ ]:
# ==================== 生成Friedman排名表格（CSV格式） ====================

def generate_friedman_ranking_table():
    """生成Friedman排名CSV表格"""
    
    table_data = []
    
    for metric in SINGLE_OBJECTIVE_METRICS:
        if metric in friedman_results:
            result = friedman_results[metric]
            
            # 创建排名数据
            rank_data = []
            for idx, alg_name in enumerate(result['alg_names']):
                rank_data.append({
                    'Metric': metric,
                    'Algorithm': ALL_ALGORITHMS[alg_name]['name'],
                    'Mean_Rank': result['mean_ranks'][idx],
                    'Type': PROPOSED_ALGORITHMS.get(alg_name, {}).get('type', 'comparison')
                })
            
            # 按排名排序
            rank_df = pd.DataFrame(rank_data)
            rank_df = rank_df.sort_values('Mean_Rank')
            rank_df['Rank'] = range(1, len(rank_df) + 1)
            
            table_data.append(rank_df)
    
    if table_data:
        combined_table = pd.concat(table_data, ignore_index=True)
        
        # 保存为CSV文件
        csv_path = '../draw/table_friedman_ranking.csv'
        combined_table.to_csv(csv_path, index=False, encoding='utf-8-sig')
        
        print("=" * 80)
        print("Friedman排名表格（CSV格式）")
        print("=" * 80)
        print(f"\n✅ CSV表格已保存到 {csv_path}")
        
        return combined_table
    
    return None

friedman_table = generate_friedman_ranking_table()
if friedman_table is not None:
    print("\n表格预览：")
    print(friedman_table.to_string(index=False))

## 6. 可扩展性分析（不同任务规模下的性能）

In [ ]:
# ==================== 不同任务规模下的性能分析 ====================

def analyze_by_task_size(metric):
    """按任务规模分析性能"""
    task_sizes = sorted(TASK_SIZE_MAP.values())
    results = {}
    
    for alg_name in ALL_ALGORITHMS.keys():
        if alg_name in merged_single_obj and metric in merged_single_obj[alg_name].columns:
            alg_results = {}
            for task_size in task_sizes:
                task_data = merged_single_obj[alg_name][
                    merged_single_obj[alg_name]['TaskSize'] == task_size
                ][metric].values
                
                if len(task_data) > 0:
                    alg_results[task_size] = {
                        'mean': np.mean(task_data),
                        'std': np.std(task_data),
                        'median': np.median(task_data)
                    }
            results[alg_name] = alg_results
    
    return results

# 分析所有指标在不同任务规模下的表现
scalability_results = {}
for metric in SINGLE_OBJECTIVE_METRICS:
    scalability_results[metric] = analyze_by_task_size(metric)

print("=" * 80)
print("不同任务规模下的性能分析")
print("=" * 80)

for metric in SINGLE_OBJECTIVE_METRICS:
    print(f"\n【{metric}】")
    print("-" * 80)
    
    # 创建数据框
    task_sizes = sorted(TASK_SIZE_MAP.values())
    data_rows = []
    
    for alg_name in ['MOPPO', 'MOPPO2', 'MOPPO2E', 'MOIPPO']:
        if alg_name in scalability_results[metric]:
            row = {'Algorithm': ALL_ALGORITHMS[alg_name]['name']}
            for task_size in task_sizes:
                if task_size in scalability_results[metric][alg_name]:
                    mean_val = scalability_results[metric][alg_name][task_size]['mean']
                    row[f'{task_size}'] = f"{mean_val:.2f}"
                else:
                    row[f'{task_size}'] = "N/A"
            data_rows.append(row)
    
    if data_rows:
        df_scalability = pd.DataFrame(data_rows)
        print(df_scalability.to_string(index=False))

In [ ]:
# ==================== 可扩展性可视化：性能随任务规模的变化 ====================

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
axes = axes.flatten()

task_sizes = sorted(TASK_SIZE_MAP.values())

for idx, metric in enumerate(SINGLE_OBJECTIVE_METRICS):
    ax = axes[idx]
    
    # 绘制改进算法
    for alg_name in ['MOPPO', 'MOPPO2', 'MOPPO2E', 'MOIPPO']:
        if alg_name in scalability_results[metric]:
            means = []
            stds = []
            valid_sizes = []
            
            for task_size in task_sizes:
                if task_size in scalability_results[metric][alg_name]:
                    means.append(scalability_results[metric][alg_name][task_size]['mean'])
                    stds.append(scalability_results[metric][alg_name][task_size]['std'])
                    valid_sizes.append(task_size)
            
            if means:
                ax.plot(valid_sizes, means, 
                       marker=ALL_ALGORITHMS[alg_name]['marker'],
                       color=ALL_ALGORITHMS[alg_name]['color'],
                       label=ALL_ALGORITHMS[alg_name]['name'],
                       linewidth=2, markersize=8)
                ax.fill_between(valid_sizes, 
                               [m - s for m, s in zip(means, stds)],
                               [m + s for m, s in zip(means, stds)],
                               alpha=0.2, color=ALL_ALGORITHMS[alg_name]['color'])
    
    ax.set_xlabel(LABELS['task_number'], fontsize=12)
    ax.set_ylabel(metric, fontsize=12)
    ax.set_title(scalability_title(metric), fontsize=14, fontweight='bold')
    ax.legend(loc='best', fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_xticks(task_sizes)

plt.tight_layout()
plt.savefig('../draw/scalability_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ 可扩展性分析图已保存")

In [ ]:
# ==================== 计算可扩展性指标：性能增长率 ====================

def compute_scalability_metrics(metric):
    """计算可扩展性指标"""
    task_sizes = sorted(TASK_SIZE_MAP.values())
    scalability_metrics = {}
    
    for alg_name in ['MOPPO', 'MOPPO2', 'MOPPO2E', 'MOIPPO']:
        if alg_name in scalability_results[metric]:
            # 获取最小和最大任务规模的数据
            min_size = min(task_sizes)
            max_size = max(task_sizes)
            
            if min_size in scalability_results[metric][alg_name] and \
               max_size in scalability_results[metric][alg_name]:
                min_mean = scalability_results[metric][alg_name][min_size]['mean']
                max_mean = scalability_results[metric][alg_name][max_size]['mean']
                
                # 计算增长率
                size_ratio = max_size / min_size
                performance_ratio = max_mean / min_mean
                
                # 可扩展性系数：如果性能增长小于规模增长，说明可扩展性好
                scalability_coefficient = performance_ratio / size_ratio
                
                scalability_metrics[alg_name] = {
                    'min_size': min_size,
                    'max_size': max_size,
                    'min_performance': min_mean,
                    'max_performance': max_mean,
                    'size_ratio': size_ratio,
                    'performance_ratio': performance_ratio,
                    'scalability_coefficient': scalability_coefficient
                }
    
    return scalability_metrics

print("=" * 80)
print("可扩展性指标分析")
print("=" * 80)
print("可扩展性系数 = 性能增长率 / 规模增长率")
print("系数越小，说明算法可扩展性越好（性能增长慢于规模增长）")
print("-" * 80)

scalability_metrics_all = {}
for metric in SINGLE_OBJECTIVE_METRICS:
    print(f"\n【{metric}】")
    print("-" * 80)
    
    metrics = compute_scalability_metrics(metric)
    scalability_metrics_all[metric] = metrics
    
    for alg_name in ['MOPPO', 'MOPPO2', 'MOPPO2E', 'MOIPPO']:
        if alg_name in metrics:
            m = metrics[alg_name]
            print(f"{ALL_ALGORITHMS[alg_name]['name']}:")
            print(f"  规模: {m['min_size']} -> {m['max_size']} (增长 {m['size_ratio']:.1f}倍)")
            print(f"  性能: {m['min_performance']:.2f} -> {m['max_performance']:.2f} (增长 {m['performance_ratio']:.2f}倍)")
            print(f"  可扩展性系数: {m['scalability_coefficient']:.4f}")
            print()

In [ ]:
# ==================== 生成可扩展性分析表格（CSV格式） ====================

def generate_scalability_table():
    """生成可扩展性分析CSV表格"""
    
    task_sizes = sorted(TASK_SIZE_MAP.values())
    table_data = []
    
    for alg_name in ['MOPPO', 'MOPPO2', 'MOPPO2E', 'MOIPPO']:
        row = {'Algorithm': ALL_ALGORITHMS[alg_name]['name']}
        
        for metric in SINGLE_OBJECTIVE_METRICS:
            if alg_name in scalability_metrics_all[metric]:
                coeff = scalability_metrics_all[metric][alg_name]['scalability_coefficient']
                row[metric] = coeff
            else:
                row[metric] = np.nan
        
        table_data.append(row)
    
    df_table = pd.DataFrame(table_data)
    
    # 保存为CSV文件
    csv_path = '../draw/table_scalability.csv'
    df_table.to_csv(csv_path, index=False, encoding='utf-8-sig')
    
    print("=" * 80)
    print("可扩展性分析表格（CSV格式）")
    print("=" * 80)
    print(f"\n✅ CSV表格已保存到 {csv_path}")
    
    return df_table

scalability_table = generate_scalability_table()
print("\n表格预览：")
print(scalability_table.to_string(index=False))

In [ ]:
# ==================== 不同任务规模下的改进率分析 ====================

def compute_improvement_by_task_size():
    """计算不同任务规模下的改进率"""
    baseline_alg = 'MOPPO'
    improved_algs = ['MOPPO2', 'MOPPO2E', 'MOIPPO']
    task_sizes = sorted(TASK_SIZE_MAP.values())
    
    improvement_data = {}
    
    for metric in SINGLE_OBJECTIVE_METRICS:
        improvement_data[metric] = {}
        
        if baseline_alg in merged_single_obj and metric in merged_single_obj[baseline_alg].columns:
            for task_size in task_sizes:
                baseline_data = merged_single_obj[baseline_alg][
                    merged_single_obj[baseline_alg]['TaskSize'] == task_size
                ][metric].values
                
                if len(baseline_data) > 0:
                    baseline_mean = np.mean(baseline_data)
                    improvement_data[metric][task_size] = {}
                    
                    for improved_alg in improved_algs:
                        if improved_alg in merged_single_obj and metric in merged_single_obj[improved_alg].columns:
                            improved_data = merged_single_obj[improved_alg][
                                merged_single_obj[improved_alg]['TaskSize'] == task_size
                            ][metric].values
                            
                            if len(improved_data) > 0:
                                improved_mean = np.mean(improved_data)
                                improvement = ((baseline_mean - improved_mean) / baseline_mean) * 100
                                improvement_data[metric][task_size][improved_alg] = improvement
    
    return improvement_data

improvement_by_size = compute_improvement_by_task_size()

print("=" * 80)
print("不同任务规模下的改进率分析（相对于MO-PPO）")
print("=" * 80)

for metric in SINGLE_OBJECTIVE_METRICS:
    print(f"\n【{metric}】")
    print("-" * 80)
    
    task_sizes = sorted(TASK_SIZE_MAP.values())
    data_rows = []
    
    for improved_alg in ['MOPPO2', 'MOPPO2E', 'MOIPPO']:
        row = {'Algorithm': ALL_ALGORITHMS[improved_alg]['name']}
        for task_size in task_sizes:
            if task_size in improvement_by_size[metric] and \
               improved_alg in improvement_by_size[metric][task_size]:
                improvement = improvement_by_size[metric][task_size][improved_alg]
                row[f'{task_size}'] = f"{improvement:.2f}%"
            else:
                row[f'{task_size}'] = "N/A"
        data_rows.append(row)
    
    if data_rows:
        df_improvement = pd.DataFrame(data_rows)
        print(df_improvement.to_string(index=False))

In [ ]:
# ==================== 改进率随任务规模变化的可视化 ====================

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
axes = axes.flatten()

task_sizes = sorted(TASK_SIZE_MAP.values())

for idx, metric in enumerate(SINGLE_OBJECTIVE_METRICS):
    ax = axes[idx]
    
    for improved_alg in ['MOPPO2', 'MOPPO2E', 'MOIPPO']:
        improvements = []
        valid_sizes = []
        
        for task_size in task_sizes:
            if task_size in improvement_by_size[metric] and \
               improved_alg in improvement_by_size[metric][task_size]:
                improvements.append(improvement_by_size[metric][task_size][improved_alg])
                valid_sizes.append(task_size)
        
        if improvements:
            ax.plot(valid_sizes, improvements,
                   marker=ALL_ALGORITHMS[improved_alg]['marker'],
                   color=ALL_ALGORITHMS[improved_alg]['color'],
                   label=ALL_ALGORITHMS[improved_alg]['name'],
                   linewidth=2, markersize=8)
    
    ax.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
    ax.set_xlabel(LABELS['task_number'], fontsize=12)
    ax.set_ylabel(LABELS['improvement_rate'], fontsize=12)
    ax.set_title(improvement_title(metric), fontsize=14, fontweight='bold')
    ax.legend(loc='best', fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_xticks(task_sizes)

plt.tight_layout()
plt.savefig('../draw/improvement_by_task_size.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ 改进率随任务规模变化图已保存")

## 5. 总结与结论

### 主要发现：

1. **改进算法性能**：MO-PPO2、MO-PPO2E和MO-IPPO相比基线算法MO-PPO的性能表现
2. **统计显著性**：通过Wilcoxon检验验证改进的统计显著性
3. **算法排名**：通过Friedman检验确定算法整体排名
4. **多目标性能**：通过HV、GD、IGD、Spread等指标评估Pareto前沿质量

### 论文写作建议：

1. **实验设置部分**：
   - 说明实验配置（10次独立运行，run_21到run_30）
   - 列出所有对比算法
   - 说明评估指标

2. **结果分析部分**：
   - 使用描述性统计表格展示结果
   - 使用箱线图展示数据分布
   - 使用统计检验验证改进的显著性
   - 使用Pareto前沿图展示多目标优化效果

3. **讨论部分**：
   - 分析改进算法的优势
   - 解释为什么某些改进有效
   - 讨论算法的适用场景

4. **结论部分**：
   - 总结主要贡献
   - 指出未来改进方向